In [ ]:
#Forecasting 
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from collections import Counter
import sys
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score
import numpy as np
from ts2vec import TS2Vec
from scipy.io import arff
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from scipy.io.arff import loadarff
import time

# Configure environment
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_SHOW_CPP_STACKTRACES'] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def window(arr, size):
    return np.array([arr[i:i + size] for i in range(len(arr) - size + 1)])

def getIdealSplit(data, min_sep=20, alpha=2):
    """Process 3D array (samples × timesteps × variables)"""
    per_sample_splits = []
    
    for sample in data:
        N, d = sample.shape
        tscs = []
        times = []
        for delta in range(5, min(500, N), 5):
            Xs = window(sample, delta)[1::delta]
            for idx, (x1, x2) in enumerate(zip(Xs[:-1], Xs[1:])):
                t = (idx + 1) * delta
                S1 = np.cov(x1, rowvar=False) + 1e-6 * np.eye(d)
                S2 = np.cov(x2, rowvar=False) + 1e-6 * np.eye(d)
                S_comb = np.cov(np.vstack((x1, x2)), rowvar=False) + 1e-6 * np.eye(d)
                logdet_S1 = np.linalg.slogdet(S1)[1]
                logdet_S2 = np.linalg.slogdet(S2)[1]
                logdet_S_comb = np.linalg.slogdet(S_comb)[1]
                logL_sep = -0.5 * (delta * logdet_S1 + delta * logdet_S2)
                logL_comb = -0.5 * (2 * delta * logdet_S_comb)
                k = d + d * (d + 1) / 2
                dBIC = -2 * (logL_comb - logL_sep) + k * np.log(2 * delta)
                tscs.append(abs(dBIC))
                times.append(t)
        tscs = np.array(tscs)
        times = np.array(times)
        mu, tau = np.mean(tscs), np.std(tscs)
        significant = times[tscs >= (mu + alpha * tau)]
        if len(significant) > 0:
            significant = sorted(significant)
            final_splits = [significant[0]]
            for t in significant[1:]:
                if t - final_splits[-1] >= min_sep:
                    final_splits.append(t)
            per_sample_splits.append(final_splits)
        else:
            per_sample_splits.append([])
    return per_sample_splits




def process_and_save_chunk_means(data, splits, dates):
    bounds = [0] + [int(s) for s in splits] + [data.shape[0]]
    data_summ = []
    date_summ = []
    for st, end in zip(bounds, bounds[1:]):
        chunk = data[st:end]
        chunk_summ = np.mean(chunk, axis=0).reshape(1,-1)
        data_summ.append(chunk_summ)
        # Get first date of chunk
        date_summ.append(dates[st])
    data_summ = np.concatenate(data_summ, axis=0)
    return data_summ, date_summ


def load_forecast_npy(name, univar=False):
    data = np.load(f'datasets/{name}.npy')    
    if univar:
        data = data[: -1:]
        
    train_slice = slice(None, int(0.6 * len(data)))
    valid_slice = slice(int(0.6 * len(data)), int(0.8 * len(data)))
    test_slice = slice(int(0.8 * len(data)), None)
    
    scaler = StandardScaler().fit(data[train_slice])
    data = scaler.transform(data)
    data = np.expand_dims(data, 0)

    pred_lens = [24, 48, 96, 288, 672]
    return data, train_slice, valid_slice, test_slice, scaler, pred_lens, 0


def _get_time_features(dt):
    return np.stack([
        dt.minute.to_numpy(),
        dt.hour.to_numpy(),
        dt.dayofweek.to_numpy(),
        dt.day.to_numpy(),
        dt.dayofyear.to_numpy(),
        dt.month.to_numpy(),
        dt.isocalendar().week.to_numpy(),
    ], axis=1).astype(float)


def load_forecast_csv(name, univar=False):
    data = pd.read_csv(f'datasets/{name}.csv', index_col='date', parse_dates=True)
    dt_embed = _get_time_features(data.index)
    n_covariate_cols = dt_embed.shape[-1]
    
    if univar:
        if name in ('ETTh1', 'ETTh2', 'ETTm1', 'ETTm2'):
            data = data[['OT']]
        elif name == 'electricity':
            data = data[['MT_001']]
        else:
            data = data.iloc[:, -1:]
        
    data = data.to_numpy()
    if name == 'ETTh1' or name == 'ETTh2':
        train_slice = slice(None, 12*30*24)
        valid_slice = slice(12*30*24, 16*30*24)
        test_slice = slice(16*30*24, 20*30*24)
    elif name == 'ETTm1' or name == 'ETTm2':
        train_slice = slice(None, 12*30*24*4)
        valid_slice = slice(12*30*24*4, 16*30*24*4)
        test_slice = slice(16*30*24*4, 20*30*24*4)
    else:
        train_slice = slice(None, int(0.6 * len(data)))
        valid_slice = slice(int(0.6 * len(data)), int(0.8 * len(data)))
        test_slice = slice(int(0.8 * len(data)), None)
    
    scaler = StandardScaler().fit(data[train_slice])
    data = scaler.transform(data)
    if name in ('electricity'):
        data = np.expand_dims(data.T, -1)  # Each variable is an instance rather than a feature
    else:
        data = np.expand_dims(data, 0)
    
    if n_covariate_cols > 0:
        dt_scaler = StandardScaler().fit(dt_embed[train_slice])
        dt_embed = np.expand_dims(dt_scaler.transform(dt_embed), 0)
        data = np.concatenate([np.repeat(dt_embed, data.shape[0], axis=0), data], axis=-1)
    
    if name in ('ETTh1', 'ETTh2', 'electricity'):
        pred_lens = [24, 48, 168, 336, 720]
    else:
        pred_lens = [24, 48, 96, 288, 672]
        
    return data, train_slice, valid_slice, test_slice, scaler, pred_lens, n_covariate_cols



def fit_svm(features, y, MAX_SAMPLES=10000):
    nb_classes = np.unique(y, return_counts=True)[1].shape[0]
    train_size = features.shape[0]

    svm = SVC(C=1e12, gamma='scale')  # Replace inf with large finite value

    if train_size // nb_classes < 5 or train_size < 50:
        return svm.fit(features, y)
    else:
        grid_search = GridSearchCV(
            svm, {
                'C': [
                    0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000,
                    1e12
                ],
                'kernel': ['rbf'],
                'degree': [3],
                'gamma': ['scale'],
                'coef0': [0],
                'shrinking': [True],
                'probability': [False],
                'tol': [0.001],
                'cache_size': [200],
                'class_weight': [None],
                'verbose': [False],
                'max_iter': [10000000],
                'decision_function_shape': ['ovr'],
                'random_state': [None]
            },
            cv=5, n_jobs=5
        )
        # If the training set is too large, subsample MAX_SAMPLES examples
        if train_size > MAX_SAMPLES:
            split = train_test_split(
                features, y,
                train_size=MAX_SAMPLES, random_state=0, stratify=y
            )
            features = split[0]
            y = split[2]
            
        grid_search.fit(features, y)
        return grid_search.best_estimator_

def fit_lr(features, y, MAX_SAMPLES=100000):
    # If the training set is too large, subsample MAX_SAMPLES examples
    if features.shape[0] > MAX_SAMPLES:
        split = train_test_split(
            features, y,
            train_size=MAX_SAMPLES, random_state=0, stratify=y
        )
        features = split[0]
        y = split[2]
        
    pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            random_state=0,
            max_iter=1000000,
            multi_class='ovr'
        )
    )
    pipe.fit(features, y)
    return pipe

def fit_knn(features, y):
    pipe = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=1)
    )
    pipe.fit(features, y)
    return pipe

def fit_ridge(train_features, train_y, valid_features, valid_y, MAX_SAMPLES=100000):
    # If the training set is too large, subsample MAX_SAMPLES examples
    if train_features.shape[0] > MAX_SAMPLES:
        split = train_test_split(
            train_features, train_y,
            train_size=MAX_SAMPLES, random_state=0
        )
        train_features = split[0]
        train_y = split[2]
    if valid_features.shape[0] > MAX_SAMPLES:
        split = train_test_split(
            valid_features, valid_y,
            train_size=MAX_SAMPLES, random_state=0
        )
        valid_features = split[0]
        valid_y = split[2]
    
    alphas = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
    valid_results = []
    for alpha in alphas:
        lr = Ridge(alpha=alpha).fit(train_features, train_y)
        valid_pred = lr.predict(valid_features)
        score = np.sqrt(((valid_pred - valid_y) ** 2).mean()) + np.abs(valid_pred - valid_y).mean()
        valid_results.append(score)
    best_alpha = alphas[np.argmin(valid_results)]
    
    lr = Ridge(alpha=best_alpha)
    lr.fit(train_features, train_y)
    return lr


def generate_pred_samples(features, data, pred_len, drop=0):
    n = data.shape[1]
    features = features[:, :-pred_len]
    labels = np.stack([ data[:, i:1+n+i-pred_len] for i in range(pred_len)], axis=2)[:, 1:]
    features = features[:, drop:]
    labels = labels[:, drop:]
    return features.reshape(-1, features.shape[-1]), \
            labels.reshape(-1, labels.shape[2]*labels.shape[3])

def cal_metrics(pred, target):
    return {
        'MSE': ((pred - target) ** 2).mean(),
        'MAE': np.abs(pred - target).mean()
    }
    
def eval_forecasting(model, data, train_slice, valid_slice, test_slice, scaler, pred_lens, n_covariate_cols):
    padding = 0
    
    t = time.time()
    all_repr = model.encode(
        data,
        causal=True,
        sliding_length=1,
        sliding_padding=0,
        batch_size=256
    )
    ts2vec_infer_time = time.time() - t
    
    train_repr = all_repr[:, train_slice]
    valid_repr = all_repr[:, valid_slice]
    test_repr = all_repr[:, test_slice]
    
    train_data = data[:, train_slice, n_covariate_cols:]
    valid_data = data[:, valid_slice, n_covariate_cols:]
    test_data = data[:, test_slice, n_covariate_cols:]
    
    ours_result = {}
    lr_train_time = {}
    lr_infer_time = {}
    out_log = {}
    for pred_len in pred_lens:
        train_features, train_labels = generate_pred_samples(train_repr, train_data, pred_len, drop=padding)
        valid_features, valid_labels = generate_pred_samples(valid_repr, valid_data, pred_len)
        test_features, test_labels = generate_pred_samples(test_repr, test_data, pred_len)
        
        t = time.time()
        lr = fit_ridge(train_features, train_labels, valid_features, valid_labels)
        lr_train_time[pred_len] = time.time() - t
        
        t = time.time()
        test_pred = lr.predict(test_features)
        lr_infer_time[pred_len] = time.time() - t

        ori_shape = test_data.shape[0], -1, pred_len, test_data.shape[2]
        test_pred = test_pred.reshape(ori_shape)
        test_labels = test_labels.reshape(ori_shape)
        
        if test_data.shape[0] > 1:
            # Multi-series case (original logic)
            test_pred_inv = scaler.inverse_transform(test_pred.swapaxes(0, 3)).swapaxes(0, 3)
            test_labels_inv = scaler.inverse_transform(test_labels.swapaxes(0, 3)).swapaxes(0, 3)
        else:
            # Single-series case (fixed logic)
            original_shape = test_pred.shape
            test_pred_2d = test_pred.reshape(-1, original_shape[-1])
            test_pred_inv = scaler.inverse_transform(test_pred_2d).reshape(original_shape)
            
            test_labels_2d = test_labels.reshape(-1, original_shape[-1])
            test_labels_inv = scaler.inverse_transform(test_labels_2d).reshape(original_shape)
            
        out_log[pred_len] = {
            'norm': test_pred,
            'raw': test_pred_inv,
            'norm_gt': test_labels,
            'raw_gt': test_labels_inv
        }
        ours_result[pred_len] = {
            'norm': cal_metrics(test_pred, test_labels),
            'raw': cal_metrics(test_pred_inv, test_labels_inv)
        }
        
    eval_res = {
        'ours': ours_result,
        'ts2vec_infer_time': ts2vec_infer_time,
        'lr_train_time': lr_train_time,
        'lr_infer_time': lr_infer_time
    }
    return out_log, eval_res


if __name__ == "__main__":
    datasets = ['ETTh1', 'ETTh2', 'ETTm1', 'electricity']
    print("Processing forecasting datasets:", ', '.join(datasets))

    # Create results directory
    if not os.path.exists("forecasting_results"):
        os.makedirs("forecasting_results")

    # Create a DataFrame to store all forecasting results
    forecast_results = pd.DataFrame(columns=['dataset', 'horizon', 'norm_mse', 'raw_mse', 'norm_mae', 'raw_mae'])

    for dataset_name in datasets:
        print(f"\nProcessing {dataset_name} dataset...")
        try:
            # Load original dataset
            if dataset_name in ['ETTh1', 'ETTh2', 'ETTm1']:
                data, train_slice, valid_slice, test_slice, scaler, pred_lens, n_covariate_cols = load_forecast_csv(dataset_name, univar=False)
                raw_df = pd.read_csv(f'datasets/{dataset_name}.csv', parse_dates=['date'])
                variable_names = raw_df.columns.tolist()[1:]
                dates = raw_df['date'].values
                train_dates = dates[train_slice]
                test_dates = dates[test_slice]
            else:
                data, train_slice, valid_slice, test_slice, scaler, pred_lens, n_covariate_cols = load_forecast_csv(dataset_name, univar=True)
                raw_df = pd.read_csv(f'datasets/{dataset_name}.csv', index_col='date', parse_dates=True)
                variable_names = raw_df.columns.tolist()
                dates = raw_df.index.values
                train_dates = dates[train_slice]
                test_dates = dates[test_slice]

            print(f"Original dataset shape for {dataset_name}: {data.shape}")
            # After loading the original CSV
            print("\nFirst 5 rows of the original raw DataFrame:")
            print(raw_df.head())  # Shows the first 5 rows of the original data

            # Chunking and splitting logic (unchanged)
            train_X = data[:, train_slice, n_covariate_cols:]
            test_X = data[:, test_slice, n_covariate_cols:]

            print("\nProcessing training data splits...")
            train_splits = getIdealSplit(train_X)
            total_train_splits = sum(len(s) for s in train_splits)
            total_train_chunks = sum(len(s)+1 for s in train_splits)
            print(f"Total splits: {total_train_splits}, Chunks: {total_train_chunks}")

            print("\nProcessing test data splits...")
            test_splits = getIdealSplit(test_X)
            total_test_splits = sum(len(s) for s in test_splits)
            total_test_chunks = sum(len(s)+1 for s in test_splits)
            print(f"Total splits: {total_test_splits}, Chunks: {total_test_chunks}")

            def process_and_save_chunk_means_with_dates(data, splits, dates):
                bounds = [0] + [int(s) for s in splits] + [data.shape[0]]
                data_summ = []
                date_summ = []
                for st, end in zip(bounds, bounds[1:]):
                    chunk = data[st:end]
                    chunk_summ = np.mean(chunk, axis=0).reshape(1,-1)
                    data_summ.append(chunk_summ)
                    date_summ.append(dates[st])
                return np.concatenate(data_summ, axis=0), date_summ

            summarized_train, date_train = [], []
            for sample_idx, (sample, splits) in enumerate(zip(train_X, train_splits)):
                print(f"Train Sample {sample_idx} splits: {splits}")
                summ, dates = process_and_save_chunk_means_with_dates(sample, splits, train_dates)
                summarized_train.append(summ)
                date_train.extend(dates)

            summarized_test, date_test = [], []
            for sample_idx, (sample, splits) in enumerate(zip(test_X, test_splits)):
                print(f"Test Sample {sample_idx} splits: {splits}")
                summ, dates = process_and_save_chunk_means_with_dates(sample, splits, test_dates)
                summarized_test.append(summ)
                date_test.extend(dates)

            # Save processed CSVs with date as index
            train_save_path = os.path.join("forecasting_results", f"{dataset_name}_TRAIN_processed.csv")
            test_save_path = os.path.join("forecasting_results", f"{dataset_name}_TEST_processed.csv")

            train_df = pd.DataFrame(np.vstack(summarized_train), columns=variable_names, index=pd.to_datetime(date_train))
            train_df.index.name = 'date'
            train_df.to_csv(train_save_path)

            test_df = pd.DataFrame(np.vstack(summarized_test), columns=variable_names, index=pd.to_datetime(date_test))
            test_df.index.name = 'date'
            test_df.to_csv(test_save_path)

            print(f"\nSaved processed data for {dataset_name}")
            print("\nFirst 5 rows of processed TRAIN DataFrame:")
            print(train_df.head())

            print("\nFirst 5 rows of processed TEST DataFrame:")
            print(test_df.head())

            # Reload processed CSV and add temporal features
            def reload_processed(path):
                df = pd.read_csv(path, index_col='date', parse_dates=True)
                dt_embed = _get_time_features(df.index)
                data = df.values
                if dt_embed.size > 0:
                    dt_scaler = StandardScaler().fit(dt_embed)
                    dt_embed = dt_scaler.transform(dt_embed)
                    data = np.concatenate([dt_embed, data], axis=1)
                return np.expand_dims(data, 0)  # Shape: (1, n_chunks, 14)

            processed_train = reload_processed(train_save_path)
            processed_test = reload_processed(test_save_path)

            print("\nReloaded processed data shapes:")
            print(f"Train: {processed_train.shape}")
            print(f"Test: {processed_test.shape}")
            print("\nFirst 5 rows of reloaded processed TRAIN data (numpy):")
            print(processed_train[0][:5])
            print("\nFirst 5 rows of reloaded processed TEST data (numpy):")
            print(processed_test[0][:5])

            # --- TS2Vec Forecasting Setup ---
            print("\nInitializing TS2Vec model...")
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            config = {
                'batch_size': 16,  # From your attached code comment
                'lr': 0.001,       # From your attached code comment
                'output_dims': 320,
                'max_train_length': 3000,
                'device': device
            }
            
            # Create model with same parameters as reference
            model = TS2Vec(
                input_dims=processed_train.shape[-1],  # Should be 14 (7 temporal + 7 vars)
                **config
            )
            
            # Train on processed chunks (shape 1, n_chunks, 14)
            print(f"Training TS2Vec on {processed_train.shape} data...")
            loss_log = model.fit(
                processed_train,  # Your reloaded (1, n_chunks, 14) array
                n_epochs=599,     # Default from reference code
                verbose=True
            )
            
            # Evaluate using your existing eval_forecasting function
            print("\nEvaluating forecasting performance...")
            out_log, eval_res = eval_forecasting(
                model=model,
                data=data,  # Original loaded data
                train_slice=train_slice,
                valid_slice=valid_slice,
                test_slice=test_slice,
                scaler=scaler,
                pred_lens=pred_lens,
                n_covariate_cols=n_covariate_cols
            )
            
            print("\nForecasting evaluation results:")
            for pred_len in pred_lens:
                print(f"Horizon {pred_len}:")
                print(f"  Norm MSE: {eval_res['ours'][pred_len]['norm']['MSE']:.4f}")
                print(f"  Raw MSE: {eval_res['ours'][pred_len]['raw']['MSE']:.4f}")
                print(f"  Norm MAE: {eval_res['ours'][pred_len]['norm']['MAE']:.4f}")
                print(f"  Raw MAE: {eval_res['ours'][pred_len]['raw']['MAE']:.4f}")
                
                # Append results to DataFrame
                new_row = {
                    'dataset': dataset_name,
                    'horizon': pred_len,
                    'norm_mse': eval_res['ours'][pred_len]['norm']['MSE'],
                    'raw_mse': eval_res['ours'][pred_len]['raw']['MSE'],
                    'norm_mae': eval_res['ours'][pred_len]['norm']['MAE'],
                    'raw_mae': eval_res['ours'][pred_len]['raw']['MAE']
                }
                forecast_results = pd.concat([forecast_results, pd.DataFrame([new_row])], ignore_index=True)

            # Save model
            model_save_path = os.path.join("forecasting_results", f"{dataset_name}_ts2vec_model.pkl")
            model.save(model_save_path)
            print(f"Saved TS2Vec model to {model_save_path}")

        except Exception as e:
            print(f"Error processing {dataset_name}: {e}")
            import traceback
            traceback.print_exc()
            continue

    # Save all forecasting results to CSV
    results_path = os.path.join("forecasting_results", "forecast_results.csv")
    forecast_results.to_csv(results_path, index=False)
    print(f"\nSaved all forecasting results to {results_path}")
    print("\nFinal forecasting results:")
    print(forecast_results)


In [ ]:
# classification with UEA
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from collections import Counter
import sys
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score
import numpy as np
from ts2vec import TS2Vec
from scipy.io import arff
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from scipy.io.arff import loadarff

# Configure environment
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_SHOW_CPP_STACKTRACES'] = "1"
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

def window(arr, size):
    return np.array([arr[i:i + size] for i in range(len(arr) - size + 1)])

def getIdealSplit(data, min_sep=20, alpha=2):
    """Process 3D array (samples × timesteps × variables)"""
    per_sample_splits = []
    
    for sample in data:
        N, d = sample.shape
        tscs = []
        times = []
        for delta in range(5, min(500, N), 5):
            Xs = window(sample, delta)[1::delta]
            for idx, (x1, x2) in enumerate(zip(Xs[:-1], Xs[1:])):
                t = (idx + 1) * delta
                S1 = np.cov(x1, rowvar=False) + 1e-6 * np.eye(d)
                S2 = np.cov(x2, rowvar=False) + 1e-6 * np.eye(d)
                S_comb = np.cov(np.vstack((x1, x2)), rowvar=False) + 1e-6 * np.eye(d)
                logdet_S1 = np.linalg.slogdet(S1)[1]
                logdet_S2 = np.linalg.slogdet(S2)[1]
                logdet_S_comb = np.linalg.slogdet(S_comb)[1]
                logL_sep = -0.5 * (delta * logdet_S1 + delta * logdet_S2)
                logL_comb = -0.5 * (2 * delta * logdet_S_comb)
                k = d + d * (d + 1) / 2
                dBIC = -2 * (logL_comb - logL_sep) + k * np.log(2 * delta)
                tscs.append(abs(dBIC))
                times.append(t)
        tscs = np.array(tscs)
        times = np.array(times)
        mu, tau = np.mean(tscs), np.std(tscs)
        significant = times[tscs >= (mu + alpha * tau)]
        if len(significant) > 0:
            significant = sorted(significant)
            final_splits = [significant[0]]
            for t in significant[1:]:
                if t - final_splits[-1] >= min_sep:
                    final_splits.append(t)
            per_sample_splits.append(final_splits)
        else:
            per_sample_splits.append([])
    return per_sample_splits





def process_and_save_chunk_means(data, splits):
  bounds = [0] + [int(s) for s in splits] + [data.shape[0]]
  data_summ = []
  for st, end in zip(bounds, bounds[1:]):
    chunk = data[st:end]
    chunk_summ = np.mean(chunk, axis=0).reshape(1,-1)
    data_summ.append(chunk_summ)
  data_summ = np.concatenate(data_summ, axis=0)
  return data_summ



def load_UEA(dataset, base_dir):
    train_path = os.path.join(base_dir, dataset, f"{dataset}_TRAIN.arff")
    test_path = os.path.join(base_dir, dataset, f"{dataset}_TEST.arff")
    train_data = loadarff(train_path)[0]
    test_data = loadarff(test_path)[0]
    
    def extract_data(data):
        res_data = []
        res_labels = []
        for t_data, t_label in data:
            t_data = np.array([d.tolist() for d in t_data])
            t_label = t_label.decode("utf-8")
            res_data.append(t_data)
            res_labels.append(t_label)
        return np.array(res_data).swapaxes(1, 2), np.array(res_labels)
    
    train_X, train_y = extract_data(train_data)
    test_X, test_y = extract_data(test_data)
    
    scaler = StandardScaler()
    scaler.fit(train_X.reshape(-1, train_X.shape[-1]))
    train_X = scaler.transform(train_X.reshape(-1, train_X.shape[-1])).reshape(train_X.shape)
    test_X = scaler.transform(test_X.reshape(-1, test_X.shape[-1])).reshape(test_X.shape)
    
    labels = np.unique(train_y)
    transform = {k: i for i, k in enumerate(labels)}
    train_y = np.vectorize(transform.get)(train_y)
    test_y = np.vectorize(transform.get)(test_y)
    return train_X, train_y, test_X, test_y

def fit_svm(features, y, MAX_SAMPLES=10000):
    nb_classes = np.unique(y, return_counts=True)[1].shape[0]
    train_size = features.shape[0]

    svm = SVC(C=1e12, gamma='scale')  # Replace inf with large finite value

    if train_size // nb_classes < 5 or train_size < 50:
        return svm.fit(features, y)
    else:
        grid_search = GridSearchCV(
            svm, {
                'C': [
                    0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000,
                    1e12
                ],
                'kernel': ['rbf'],
                'degree': [3],
                'gamma': ['scale'],
                'coef0': [0],
                'shrinking': [True],
                'probability': [False],
                'tol': [0.001],
                'cache_size': [200],
                'class_weight': [None],
                'verbose': [False],
                'max_iter': [10000000],
                'decision_function_shape': ['ovr'],
                'random_state': [None]
            },
            cv=5, n_jobs=5
        )
        # If the training set is too large, subsample MAX_SAMPLES examples
        if train_size > MAX_SAMPLES:
            split = train_test_split(
                features, y,
                train_size=MAX_SAMPLES, random_state=0, stratify=y
            )
            features = split[0]
            y = split[2]
            
        grid_search.fit(features, y)
        return grid_search.best_estimator_

def fit_lr(features, y, MAX_SAMPLES=100000):
    # If the training set is too large, subsample MAX_SAMPLES examples
    if features.shape[0] > MAX_SAMPLES:
        split = train_test_split(
            features, y,
            train_size=MAX_SAMPLES, random_state=0, stratify=y
        )
        features = split[0]
        y = split[2]
        
    pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            random_state=0,
            max_iter=1000000,
            multi_class='ovr'
        )
    )
    pipe.fit(features, y)
    return pipe

def fit_knn(features, y):
    pipe = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=1)
    )
    pipe.fit(features, y)
    return pipe

def fit_ridge(train_features, train_y, valid_features, valid_y, MAX_SAMPLES=100000):
    # If the training set is too large, subsample MAX_SAMPLES examples
    if train_features.shape[0] > MAX_SAMPLES:
        split = train_test_split(
            train_features, train_y,
            train_size=MAX_SAMPLES, random_state=0
        )
        train_features = split[0]
        train_y = split[2]
    if valid_features.shape[0] > MAX_SAMPLES:
        split = train_test_split(
            valid_features, valid_y,
            train_size=MAX_SAMPLES, random_state=0
        )
        valid_features = split[0]
        valid_y = split[2]
    
    alphas = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
    valid_results = []
    for alpha in alphas:
        lr = Ridge(alpha=alpha).fit(train_features, train_y)
        valid_pred = lr.predict(valid_features)
        score = np.sqrt(((valid_pred - valid_y) ** 2).mean()) + np.abs(valid_pred - valid_y).mean()
        valid_results.append(score)
    best_alpha = alphas[np.argmin(valid_results)]
    
    lr = Ridge(alpha=best_alpha)
    lr.fit(train_features, train_y)
    return lr


def eval_classification(model, train_data, train_labels, test_data, test_labels, eval_protocol='linear'):
    assert train_labels.ndim == 1 or train_labels.ndim == 2
    train_repr = model.encode(train_data, encoding_window='full_series' if train_labels.ndim == 1 else None)
    test_repr = model.encode(test_data, encoding_window='full_series' if train_labels.ndim == 1 else None)

    if eval_protocol == 'linear':
        fit_clf = fit_lr
    elif eval_protocol == 'svm':
        fit_clf = fit_svm
    elif eval_protocol == 'knn':
        fit_clf = fit_knn
    else:
        assert False, 'unknown evaluation protocol'

    def merge_dim01(array):
        return array.reshape(array.shape[0]*array.shape[1], *array.shape[2:])

    if train_labels.ndim == 2:
        train_repr = merge_dim01(train_repr)
        train_labels = merge_dim01(train_labels)
        test_repr = merge_dim01(test_repr)
        test_labels = merge_dim01(test_labels)

    clf = fit_clf(train_repr, train_labels)

    acc = clf.score(test_repr, test_labels)
    if eval_protocol == 'linear':
        y_score = clf.predict_proba(test_repr)
    else:
        y_score = clf.decision_function(test_repr)
    test_labels_onehot = label_binarize(test_labels, classes=np.arange(train_labels.max()+1))
    auprc = average_precision_score(test_labels_onehot, y_score)
    
    return y_score, { 'acc': acc, 'auprc': auprc }
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def reload_processed_csv(name, split='train'):
    """
    Reloads processed train or test CSV data, standardizes using train stats, and reshapes.

    Args:
        name (str): Dataset name (expects files like processed_dataset/{name}_TRAIN_processed.csv).
        split (str): 'train' or 'test'.

    Returns:
        data (np.ndarray): Array of shape [instance, timestep, feature].
        scaler (StandardScaler): Fitted scaler (on train data).
    """
    # Step 1: Load processed CSV file with correct path and name
    train_path = f'processed_dataset/{name}_TRAIN_processed.csv'
    test_path = f'processed_dataset/{name}_TEST_processed.csv'

    train_data = pd.read_csv(train_path)
    test_data = pd.read_csv(test_path)

    # Remove label column before scaling
    train_features = train_data.drop('label', axis=1).values
    test_features = test_data.drop('label', axis=1).values

    # Step 2: No splitting needed, already split

    # Step 3: Standardize using train stats
    scaler = StandardScaler().fit(train_features)
    if split == 'train':
        data = scaler.transform(train_features)
    elif split == 'test':
        data = scaler.transform(test_features)
    else:
        raise ValueError("split must be 'train' or 'test'")

    # Step 4: Reshape the data
    # You can adjust the logic below as needed for your datasets
    if name in ('dataset',):  # Add other dataset names as needed
        data = np.expand_dims(data.T, -1)
    else:
        data = np.expand_dims(data, 0)

    # Step 5: Return
    return data, scaler

if __name__ == "__main__":
    root_dir = "/media/DiskDrive1/Datasets"
    multivariate_dir = os.path.join(root_dir, "Multivariate_arff")
    datasets = sorted([
        d for d in os.listdir(multivariate_dir)
        if os.path.isdir(os.path.join(multivariate_dir, d))
    ])
    print(f"Found {len(datasets)} UEA multivariate datasets.")

    if not os.path.exists("processed_dataset"):
        os.makedirs("processed_dataset")
    results = []


    for dataset_name in datasets:
        print(f"\nProcessing {dataset_name} UEA dataset...")
        try:
            train_X, train_y, test_X, test_y = load_UEA(dataset_name, multivariate_dir)
        except Exception as e:
            print(f"Error loading {dataset_name}: {e}")
            continue
        #print(f"Original train_X shape: {train_X.shape}")
        #print(f"Original test_X shape: {test_X.shape}")

        #print(train_y)
        # Process train data with per-sample splits
        splits = getIdealSplit(train_X)
        total_splits = sum(len(s) for s in splits)
        total_chunks = sum(len(s)+1 for s in splits)
        #print(f"\nTotal splits across all samples: {total_splits}")
        #print(f"Total chunks across all samples: {total_chunks}")

        summarized_data = []
        for sample_idx, (sample, sample_splits) in enumerate(zip(train_X, splits)):
            #print(f"Sample {sample_idx} splits: {sample_splits}")
            sample_summary = process_and_save_chunk_means(sample, sample_splits)
            summarized_data.append(sample_summary)
        test_splits = getIdealSplit(test_X)
        test_summarized_data = []
        for sample_idx, (sample, sample_splits) in enumerate(zip(test_X, test_splits)):
            sample_summary = process_and_save_chunk_means(sample, sample_splits)
            test_summarized_data.append(sample_summary)

        # Create DataFrames with proper label alignment
        train_save_path = os.path.join("processed_dataset", f"{dataset_name}_TRAIN_processed.csv")
        test_save_path = os.path.join("processed_dataset", f"{dataset_name}_TEST_processed.csv")
        train_df = pd.DataFrame(np.vstack(summarized_data))
        train_df['label'] = np.repeat(train_y, [s.shape[0] for s in summarized_data])
        #print("\nProcessed Training Labels:")
        #print(train_df['label'].values)
        test_df = pd.DataFrame(np.vstack(test_summarized_data))
        test_df['label'] = np.repeat(test_y, [s.shape[0] for s in test_summarized_data])
        pd.set_option('display.max_columns', None)
        #print("\nProcessed Training Data Preview:")
        #print(f"Shape: {train_df.shape} (rows: chunks, columns: features + label)")
        #print("First 5 rows:")
        #print(train_df.head())

        #print("\nProcessed Test Data Preview:")
        #print(f"Shape: {test_df.shape} (rows: chunks, columns: features + label)")
        #print("First 5 rows:")
        #print(test_df.head())
        pd.reset_option('display.max_columns')
        #print(f"\nTrain processed data: {len(train_df)} rows, {train_df['label'].nunique()} unique labels.")
        #print(f"Total labels in train processed data: {train_df['label'].count()}")
        #print(f"\nTest processed data: {len(test_df)} rows, {test_df['label'].nunique()} unique labels.")
        #print(f"Total labels in test processed data: {test_df['label'].count()}")

        # Save processed data
        train_df.to_csv(train_save_path, index=False)
        test_df.to_csv(test_save_path, index=False)
        print(f"Saved processed data for {dataset_name}")
        
        # Reload processed data
        reloaded_train_data, scaler = reload_processed_csv(dataset_name, split='train')
        print(f"Reloaded processed train data shape: {reloaded_train_data.shape}")
        reloaded_test_data, _ = reload_processed_csv(dataset_name, split='test')
        print(f"Reloaded processed test data shape: {reloaded_test_data.shape}")

        # ==================== TS2Vec Classification ====================
        try:
            # Ensure 3D input shape [num_instances, num_timesteps, num_features]
            train_input = reloaded_train_data
            test_input = reloaded_test_data
            
            # Initialize TS2Vec model
            #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model = TS2Vec(
                input_dims=train_input.shape[-1],
                device=device,
                batch_size=32,
                lr=0.005,
                output_dims=320,
                max_train_length=3000
            )
            
            # Train model
            print(f"\nTraining TS2Vec on {dataset_name}...")
            loss_log = model.fit(
                train_input,
                n_epochs=300,
                verbose=True
            )
            
            # Generate representations
            print("\nGenerating representations...")
            train_repr = model.encode(train_input, encoding_window=1).squeeze(0)  # Shape: (713, 320)
            test_repr = model.encode(test_input, encoding_window=1).squeeze(0)    # Shape: (771, 320)
            
            # Prepare labels
            train_labels = train_df['label'].values
            test_labels = test_df['label'].values
            
            # Ensure matching shapes
            assert train_repr.shape[0] == len(train_labels), \
                f"Train repr: {train_repr.shape[0]} vs labels: {len(train_labels)}"
            assert test_repr.shape[0] == len(test_labels), \
                f"Test repr: {test_repr.shape[0]} vs labels: {len(test_labels)}"
            
            # Train SVM classifier
            print("\nTraining SVM classifier...")
            clf = fit_svm(train_repr, train_labels)
            
            # Evaluate
            acc = clf.score(test_repr, test_labels)
            print(f"\n{dataset_name} Classification Accuracy: {acc:.4f}")
            print(f"Final TS2Vec accuracy for {dataset_name}: {acc:.2%}")
            results.append({'dataset': dataset_name, 'accuracy': acc})
        except Exception as e:
            print(f"Error in TS2Vec classification for {dataset_name}: {str(e)}")
        # ================================================================

        print(f"\nCompleted processing for {dataset_name}")
        print("="*80 + "\n")
    # Save results to UEA_MEAN.csv
    pd.DataFrame(results).to_csv('UEA_MEAN1.csv', index=False)
    print("Saved all accuracies to UEA_MEAN.csv")

    print("All datasets processed!")
